In [1]:
from datasets import load_dataset
import numpy as np
import evaluate
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaModel, RobertaForSequenceClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "roberta-base"

/home/lewy700/Documents/roberta-hypernet-custom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = RobertaTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=512)

dataset = load_dataset("glue", "cola")

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [3]:
# --- Hypernetwork: A -> B ---
class LoRAHyperNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, lora_dim):
        super().__init__()
        self.fc1 = nn.Linear(lora_dim * input_dim * 2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, input_dim * 2 * lora_dim)

    def forward(self, inpt):
        flat = inpt.view(-1)
        h = torch.relu(self.fc1(flat))
        opt = self.fc2(h).view(inpt.shape)  # B shape: [in_dim, r]
        return (opt[0], opt[1].T)

In [4]:
class DynamicLoRAQuery(nn.Module):
    def __init__(self, original_linear: nn.Linear, hypernet: nn.Module, lora_r: int):
        super().__init__()
        self.original = original_linear  # query projection: nn.Linear
        self.hypernet = hypernet
        self.hidden_size = original_linear.in_features
        self.lora_r = lora_r

        # Fixed or random A
        self.register_buffer("A_fixed", torch.randn(lora_r, self.hidden_size))

    def forward(self, x, training=True):
        A = torch.randn(self.lora_r, self.hidden_size, device=x.device) if training else self.A_fixed
        B = self.hypernet(A)  # [hidden, r]
        W_lora = torch.matmul(B, A)  # [hidden, hidden]

        # Apply original projection + LoRA residual
        out = self.original(x) + torch.matmul(x, W_lora.T)
        return out


In [13]:
class RobertaWithDynamicLoRA(RobertaForSequenceClassification):
    def __init__(self, config, lora_r=8, hypernet_hidden_dim=128):
        super().__init__(config)
        self.config.output_hidden_states = True 
        self.hidden_size = config.hidden_size
        self.lora_r = lora_r

        # Create hypernetworks
        self.hypernet_first = LoRAHyperNet(self.hidden_size, hypernet_hidden_dim, lora_r)
        self.hypernet_last = LoRAHyperNet(self.hidden_size, hypernet_hidden_dim, lora_r)

        # Replace query projection in first and last layers with LoRA-injected version
        first_attn = self.roberta.encoder.layer[0].attention.self
        last_attn = self.roberta.encoder.layer[-1].attention.self

        first_attn.query = DynamicLoRAQuery(first_attn.query, self.hypernet_first, lora_r)
        last_attn.query = DynamicLoRAQuery(last_attn.query, self.hypernet_last, lora_r)

        self.dropout = nn.Dropout(config.hidden_dropout_prob)


In [14]:
# class RobertaWithDynamicLoRA(RobertaForSequenceClassification):
#     def __init__(self, config, lora_r=1, hypernet_hidden_dim=128):
#         super().__init__(config)
#         self.config.output_hidden_states = True 
#         self.hidden_size = config.hidden_size
#         self.lora_r = lora_r

#         # Create hypernetwork
#         self.hypernet = LoRAHyperNet(self.hidden_size, hypernet_hidden_dim, lora_r)

#         # Use fixed A in eval, learned through computation in training
#         self.register_buffer("A_fixed", torch.randn(lora_r, self.hidden_size))

#         self.dropout = nn.Dropout(config.hidden_dropout_prob)

#     def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
#         device = input_ids.device
        
#         print(self.hypernet.fc1.weight[:5][:5]) # check if the gradients flow properly throught the hypernetwork
        
#         init_A_B = torch.randn((2, self.lora_r, self.hidden_size), device=device) # [r, hidden_size * 2]
#         A, B = self.hypernet(init_A_B)

#         print(A.shape, B.shape)

#         # Forward pass through model to get hidden states
#         outputs = self.roberta(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             output_hidden_states=True,
#             return_dict=True
#         )
#         hidden_states = outputs.hidden_states[-1]  # [batch, seq_len, hidden_size]

#         # Apply dynamic LoRA on final layer hidden states
#         # hidden + (hidden @ A.T @ B.T)
#         lora_out = torch.matmul(hidden_states, A.T)     # [batch, seq_len, r]
#         lora_out = torch.matmul(lora_out, B.T)          # [batch, seq_len, hidden_size]
#         adapted_hidden = hidden_states + lora_out

#         # Use [CLS] token from adapted hidden
#         cls_output = adapted_hidden[:, 0]  # [batch, hidden]
#         logits = self.dropout(cls_output)
#         logits = self.classifier.out_proj(logits)

#         loss = None
#         if labels is not None:
#             loss_fn = torch.nn.CrossEntropyLoss()
#             loss = loss_fn(logits, labels)

#         return {"logits": logits, "loss": loss} if loss is not None else {"logits": logits}


In [15]:
base_model = RobertaWithDynamicLoRA.from_pretrained(model_name)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=base_model.lora_r,
    lora_alpha=8,
    lora_dropout=0.1
)

base_model = get_peft_model(base_model, peft_config=peft_config)

for param in base_model.hypernet.parameters():
    param.requires_grad = True

base_model.roberta.encoder.layer[-1].attention.self.query.lora_A.default.weight.requires_grad = False
base_model.roberta.encoder.layer[-1].attention.self.query.lora_B.default.weight.requires_grad = False

total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Some weights of RobertaWithDynamicLoRA were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'hypernet_first.fc1.bias', 'hypernet_first.fc1.weight', 'hypernet_first.fc2.bias', 'hypernet_first.fc2.weight', 'hypernet_last.fc1.bias', 'hypernet_last.fc1.weight', 'hypernet_last.fc2.bias', 'hypernet_last.fc2.weight', 'roberta.encoder.layer.0.attention.self.query.A_fixed', 'roberta.encoder.layer.0.attention.self.query.hypernet.fc1.bias', 'roberta.encoder.layer.0.attention.self.query.hypernet.fc1.weight', 'roberta.encoder.layer.0.attention.self.query.hypernet.fc2.bias', 'roberta.encoder.layer.0.attention.self.query.hypernet.fc2.weight', 'roberta.encoder.layer.0.attention.self.query.original.bias', 'roberta.encoder.layer.0.attention.self.query.original.weight', 'roberta.encoder.layer.11.attention.self.query.A_fixed', 'roberta.encoder.layer.11.attenti

ValueError: Target module DynamicLoRAQuery(
  (original): Linear(in_features=768, out_features=768, bias=True)
  (hypernet): LoRAHyperNet(
    (fc1): Linear(in_features=12288, out_features=128, bias=True)
    (fc2): Linear(in_features=128, out_features=12288, bias=True)
  )
) is not supported. Currently, only the following modules are supported: `torch.nn.Linear`, `torch.nn.Embedding`, `torch.nn.Conv1d`, `torch.nn.Conv2d`, `torch.nn.Conv3d`, `transformers.pytorch_utils.Conv1D`, `torch.nn.MultiheadAttention.`.

In [ ]:
for name, param in base_model.named_parameters():
    if "hypernet" in name:
        print(name, param.requires_grad)

base_model.model.hypernet.fc1.weight True
base_model.model.hypernet.fc1.bias True
base_model.model.hypernet.fc2.weight True
base_model.model.hypernet.fc2.bias True


In [ ]:
print()

In [ ]:
training_args = TrainingArguments(
    output_dir="./outputs/reoberta_base_cola",
    eval_strategy="epoch",
    # eval_steps=25,
    save_strategy="steps",
    save_steps=1000000000,
    learning_rate=4e-4,
    per_device_train_batch_size=16, # 16
    gradient_accumulation_steps=2, # 2
    per_device_eval_batch_size=32,
    num_train_epochs=80, # 80
    logging_dir="./logs/roberta_base_cola",
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    disable_tqdm=False,
    # remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

tensor([[-0.0077, -0.0014,  0.0109,  ..., -0.0021,  0.0133, -0.0108],
        [ 0.0046, -0.0033, -0.0094,  ..., -0.0258, -0.0208,  0.0098],
        [-0.0384,  0.0216, -0.0093,  ..., -0.0359,  0.0092, -0.0145],
        [ 0.0195, -0.0014, -0.0101,  ...,  0.0456, -0.0057, -0.0004],
        [-0.0076, -0.0065, -0.0051,  ...,  0.0095, -0.0265, -0.0421]],
       device='cuda:0', grad_fn=<SliceBackward0>)
torch.Size([1, 768]) torch.Size([768, 1])
tensor([[-0.0077, -0.0014,  0.0109,  ..., -0.0021,  0.0133, -0.0108],
        [ 0.0046, -0.0033, -0.0094,  ..., -0.0258, -0.0208,  0.0098],
        [-0.0384,  0.0216, -0.0093,  ..., -0.0359,  0.0092, -0.0145],
        [ 0.0195, -0.0014, -0.0101,  ...,  0.0456, -0.0057, -0.0004],
        [-0.0076, -0.0065, -0.0051,  ...,  0.0095, -0.0265, -0.0421]],
       device='cuda:0', grad_fn=<SliceBackward0>)
torch.Size([1, 768]) torch.Size([768, 1])


Epoch,Training Loss,Validation Loss


tensor([[-0.0077, -0.0014,  0.0109,  ..., -0.0021,  0.0133, -0.0108],
        [ 0.0046, -0.0033, -0.0094,  ..., -0.0258, -0.0208,  0.0098],
        [-0.0384,  0.0216, -0.0093,  ..., -0.0359,  0.0092, -0.0145],
        [ 0.0195, -0.0014, -0.0101,  ...,  0.0456, -0.0057, -0.0004],
        [-0.0076, -0.0065, -0.0051,  ...,  0.0095, -0.0265, -0.0421]],
       device='cuda:0', grad_fn=<SliceBackward0>)
torch.Size([1, 768]) torch.Size([768, 1])
tensor([[-0.0077, -0.0014,  0.0109,  ..., -0.0021,  0.0133, -0.0108],
        [ 0.0046, -0.0033, -0.0094,  ..., -0.0258, -0.0208,  0.0098],
        [-0.0384,  0.0216, -0.0093,  ..., -0.0359,  0.0092, -0.0145],
        [ 0.0195, -0.0014, -0.0101,  ...,  0.0456, -0.0057, -0.0004],
        [-0.0076, -0.0065, -0.0051,  ...,  0.0095, -0.0265, -0.0421]],
       device='cuda:0', grad_fn=<SliceBackward0>)
torch.Size([1, 768]) torch.Size([768, 1])
tensor([[-0.0077, -0.0014,  0.0109,  ..., -0.0021,  0.0133, -0.0108],
        [ 0.0046, -0.0033, -0.0094,  ..., -0

KeyboardInterrupt: 